In [5]:
#Read in data with TEST fingerprints included
import pandas as pd
data = pd.read_csv('../src/data/LD50_pre-grid-search_and_TEST.csv').set_index("Unnamed: 0")

In [34]:
from sklearn.utils.validation import check_array
import numpy as np

#Import things that will be affected by these functions
from sklearn.model_selection import train_test_split
from genra.rax.skl.hybrid import GenRAPredValueHybrid
from numpy import sqrt
from sklearn.metrics import r2_score

#GenRA currrently does not support single-sample predictions for metrics other than the binary Jaccard metric,
# so this code will circumvent that with small alterations to the source code. 
def kneighbors_sim(self,X):
    """
    Find the k-nearest neighbours for each instance and similarity scores. 
    All distances (D) are converted to similarity (S) by:
    
                D - D.min()
    Sim =   --------------
            D.max()-D.min()
    We assume D.min()==0

    """
    neigh_dist, neigh_ind = self.kneighbors(X)
    
    """If the metric used has a max of 1, use this commented-out version of the following code. For metrics
    with larger possible metrics (esp. canberra), you will need to restore this to the edited form without the 
    comment blocks"""
    
    # Convert distances to similarities:
    # if self.metric == 'jaccard':
    neigh_sim = 1-neigh_dist
    # else:
    #     if neigh_dist.max() > 0:
    #         neigh_dist_n = neigh_dist / neigh_dist.max()
    #         neigh_sim = 1 - neigh_dist_n
    #     else:
    #         neigh_sim = 1
                    
    
    return neigh_sim, neigh_ind

def predict(self, X):
    """Predict the target for the provided data

    Parameters
    ----------
    X : array-like, shape (n_queries, n_features), \
            or (n_queries, n_indexed) if metric == 'precomputed'
        Test samples.

    Returns
    -------
    y : array of int, shape = [n_queries] or [n_queries, n_outputs]
        Target values
    """
    X = check_array(X, accept_sparse='csr')

    neigh_sim, neigh_ind = kneighbors_sim(self,X)
    
    _y = self._y
    if _y.ndim == 1:
        _y = _y.reshape((-1, 1))

    y_pred = np.empty((X.shape[0], _y.shape[1]), dtype=np.float64)

    denom=np.sum(neigh_sim)

    for j in range(_y.shape[1]):
        num = np.sum(_y[neigh_ind, j] * neigh_sim, axis=1)
        if denom > 0:
            y_pred[:, j] = num / denom
        else:
            denom = len(neigh_ind)
            y_pred[:, j] = num / denom
            
    if self._y.ndim == 1:
        y_pred = y_pred.ravel()

    return y_pred

In [35]:
#Here we create a generalized Jaccard metric, which has been tested for consistency with the pre-built
#Jaccard metric on binary sets

def generalJaccard(row1, row2):
    diff = np.array(row1)-np.array(row2)
    denom = (np.dot(diff, diff)+np.dot(row1, row2))
    if denom != 0:
        similarity = np.dot(row1, row2)/(np.dot(diff, diff)+np.dot(row1, row2))
    else:
        similarity = 0
    return similarity

def generalJaccardDistance(row1, row2):
    similarity = generalJaccard(row1, row2)
    distance = 1 - similarity
    return distance

In [42]:
#Test the performance of the TEST fingerprints alone or Morgan, for comparison
scores = {"state":[], "r2":[], 'rmse':[]}
for state in [4745,134,907, 654, 607, 90, 9054,9,12]:
    slices = [slice(0,729),slice(729,2777), slice(2777,4825), slice(4825, 5727), slice(5727, None)]
    x_train, x_test, y_train, y_test = train_test_split(data.iloc[:,1:], data.iloc[:,0], random_state=state)
    tester = GenRAPredValueHybrid(n_neighbors =8, slices = slices, hybrid_weights=[0,100,0,0,0],metric = 'cosine')
    y_preds = []

    for x in x_test.iterrows():
        tester.fit(data.loc[data.index != x[0]].iloc[:, 1:], data.loc[data.index != x[0]].iloc[:,0])
        y_preds.append(predict(tester,list(x[1:])))
    r2 = r2_score(y_test, y_preds)
    rmse = sqrt((sum([(y_preds[i]-y_test.values[i])**2 for i in range(0,len(y_preds))])/len(y_preds)))
    scores['r2'].append(r2)
    scores['rmse'].append(rmse)
    scores['state'].append(state)
    scores_df = pd.DataFrame(scores)
    scores_df.to_csv('metric_test_TEST_cosine_Morgan.csv')